In [ ]:


from pathlib import Path
import pandas as pd


path_sdr = Path(r'C:\Users\jfontes\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Products\Small Data Requests\2025\DavisPop')


def clean_fips(df):
    
    if 'Block Group ID' in df.columns:                   df['Block Group ID'                  ] = df['Block Group ID'                  ].astype(str)
    if 'Tract ID' in df.columns:                         df['Tract ID'                        ] = df['Tract ID'                        ].astype(str).apply('{:0>6}'.format)
    if 'STATEFP' in df.columns:                          df['STATEFP'                         ] = df['STATEFP'                         ].astype(str).apply('{:0>2}'.format)
    if 'State FIPS' in df.columns:                       df['State FIPS'                      ] = df['State FIPS'                      ].astype(str).apply('{:0>2}'.format)
    if 'Place ID' in df.columns:                         df['Place ID'                        ] = df['Place ID'                        ].astype(str).apply('{:0>5}'.format)
    if 'COUNTYFP' in df.columns:                         df['COUNTYFP'                        ] = df['COUNTYFP'                        ].astype(str).apply('{:0>3}'.format)
    if 'County FIPS' in df.columns:                      df['County FIPS'                     ] = df['County FIPS'                     ].astype(str).apply('{:0>3}'.format)
    if 'Congressional District' in df.columns:           df['Congressional District'          ] = df['Congressional District'          ].astype(str).apply('{:0>2}'.format)
    if 'State Legislative Upper District' in df.columns: df['State Legislative Upper District'] = df['State Legislative Upper District'].astype(str).apply('{:0>3}'.format)
    if 'State Legislative Lower District' in df.columns: df['State Legislative Lower District'] = df['State Legislative Lower District'].astype(str).apply('{:0>3}'.format)

    return df

def remove_post(x, exp='.'):
    try: x = x.split(exp, 1)[0]
    except: pass
    return x



In [ ]:


# CDP
df_pop_cdp = pd.read_excel(path_sdr / 'orig' / 'Davis_Pop Places DEC_original.xlsx')
df_davis_pop_cdp = df_pop_cdp[df_pop_cdp['NAME'] == 'Davis']

display(df_davis_pop_cdp.head())
print(df_davis_pop_cdp.Population.sum())



In [ ]:


# Block Groups
df_pop_block_groups = pd.read_excel(path_sdr / 'orig' / 'Davis_Pop Block Groups DEC_original.xlsx')
df_pop_block_groups = clean_fips(df_pop_block_groups)
df_pop_block_groups['BG'] = df_pop_block_groups['State FIPS'] + df_pop_block_groups['County FIPS'] + df_pop_block_groups['Tract ID'] + df_pop_block_groups['Block Group ID']


df_davis_blocks = pd.read_csv(path_sdr / 'orig' / 'Davis_Blocks.csv', dtype=str)
davis_bg = df_davis_blocks['BG'].unique()
davis_bg

df_davis_pop_bg = df_pop_block_groups[df_pop_block_groups['BG'].isin(davis_bg)]
df_davis_pop_bg = df_davis_pop_bg.groupby(['Year', 'Variable'], as_index=False).agg(Population=('Population', 'sum'))

display(df_davis_pop_bg.head())
print(df_davis_pop_bg.Population.sum())



In [ ]:


# Tracts
df_pop_tracts = pd.read_excel(path_sdr / 'orig' / 'Davis_Pop Tracts DEC_original.xlsx')


df_davis_blocks = pd.read_csv(path_sdr / 'orig' / 'Davis_Blocks.csv', dtype=str)
df_davis_blocks['CTBNA'] = df_davis_blocks['CTBNA'].apply(remove_post)
df_davis_blocks['CTBNA'] = df_davis_blocks['CTBNA'].astype('int64')
davis_tracts = df_davis_blocks['CTBNA'].unique()


df_davis_pop_tract = df_pop_tracts[df_pop_tracts['Tract ID'].isin(davis_tracts)]
df_davis_pop_tract = df_davis_pop_tract.groupby(['Year', 'Variable'], as_index=False).agg(Population=('Population', 'sum'))

display(df_davis_pop_tract.head())
print(df_davis_pop_tract.Population.sum())



In [ ]:
def clean_df(df):

    df = df.rename(columns={'Variable':'Age Group'})
    df = df[['Age Group', 'Population']].drop_duplicates()
    df['Age Group'] = df['Age Group'].str.replace('Total ', '')
    df['sort'] = pd.Categorical(df['Age Group'], [
        'Female Under 5 years'
            , 'Female 5 to 9 years'
            , 'Female 10 to 14 years'
            , 'Female 15 to 19 years'
            , 'Female 20 to 24 years'
            , 'Female 25 to 29 years'
            , 'Female 30 to 34 years'
            , 'Female 35 to 39 years'
            , 'Female 40 to 44 years'
            , 'Female 45 to 49 years'
            , 'Female 50 to 54 years'
            , 'Female 55 to 59 years'
            , 'Female 60 to 64 years'
            , 'Female 65 to 69 years'
            , 'Female 70 to 74 years'
            , 'Female 75 to 79 years'
            , 'Female 80 to 84 years'
            , 'Female 85 years and over'
            , 'Male Under 5 years'
            , 'Male 5 to 9 years'
            , 'Male 10 to 14 years'
            , 'Male 15 to 19 years'
            , 'Male 20 to 24 years'
            , 'Male 25 to 29 years'
            , 'Male 30 to 34 years'
            , 'Male 35 to 39 years'
            , 'Male 40 to 44 years'
            , 'Male 45 to 49 years'
            , 'Male 50 to 54 years'
            , 'Male 55 to 59 years'
            , 'Male 60 to 64 years'
            , 'Male 65 to 69 years'
            , 'Male 70 to 74 years'
            , 'Male 75 to 79 years'
            , 'Male 80 to 84 years'
            , 'Male 85 years and over'
    ])

    df = df.sort_values('sort').reset_index(drop=True).drop('sort', axis=1)

    return df



df_davis_pop_cdp   = clean_df(df_davis_pop_cdp  )
df_davis_pop_bg    = clean_df(df_davis_pop_bg   )
df_davis_pop_tract = clean_df(df_davis_pop_tract)




In [ ]:


# with pd.ExcelWriter(path_sdr / 'Davis Population_2000 DEC.xlsx', engine='xlsxwriter') as writer:
#     df_davis_pop_cdp  .to_excel(writer, index=False, sheet_name='CDP'  )
#     df_davis_pop_bg   .to_excel(writer, index=False, sheet_name='BG'   )
#     df_davis_pop_tract.to_excel(writer, index=False, sheet_name='Tract')

